# Colorectal Cancer Image Classification with Machine Learning 
### Comparison of Random Forest, MLP and CNN performance

> **Group project.** Completed as a group assignment for a Master of Computer Science machine learning unit at the University of Sydney (2026). See the README for the project overview and contributions.


## Setup and dependencies

Uses the libraries NumPy, pandas, scikit-learn [9] and Keras [10] (the available **PyTorch backend**, set before importing Keras). The models are written directly with `keras.Sequential`, not pre-trained networks. `RUN_SEARCHES=False` reloads the recorded validation results without repeating the expensive search; the final fit cells still train fresh models.

Required packages: NumPy, pandas, Matplotlib, scikit-learn, PyTorch and Keras 3. The notebook was run with NumPy 2.5.4, pandas 3.1.0, Matplotlib 3.11.2, scikit-learn 1.9.1, PyTorch 2.14.1 and Keras 3.15.1.

In [ ]:
# Dependencies (install once if missing, then restart the kernel):
# %pip install numpy pandas matplotlib scikit-learn torch keras

In [ ]:
# --- Environment: choose the Keras backend and keep matplotlib's cache local ---
import os
os.environ["KERAS_BACKEND"] = "torch"  # Configure before importing Keras (Keras 3 supports several backends)
# Matplotlib's font cache goes inside the project folder, which avoids permission problems.
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".mplconfig"))

# --- Libraries: standard library, then NumPy/pandas/plotting, then machine learning ---
from pathlib import Path
import time
import json
import random
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import keras
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from IPython.display import display

pd.set_option("display.max_columns", None)  # show every hyperparameter column
pd.set_option("display.width", 200)

# --- Reproducibility: one seed for every random number generator we use ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)
# Cap CPU threads so run times are comparable and the machine stays responsive.
torch.set_num_threads(min(4, os.cpu_count() or 1))

# --- Output folder and hyperparameter-search switch ---
# Saved tables and figures go in Results/ (created if it does not exist).
RESULTS_DIR = Path("Results")
RESULTS_DIR.mkdir(exist_ok=True)
# Hyperparameter-search switch. True: run the three searches and save their results to CSV.
# False: load the saved CSVs instead (a search is also run if its CSV is missing).
RUN_SEARCHES = False
RF_SEARCH_CSV = RESULTS_DIR / "rf_search.csv"
MLP_SEARCH_CSV = RESULTS_DIR / "mlp_search.csv"
CNN_SEARCH_CSV = RESULTS_DIR / "cnn_search.csv"

# PathMNIST label order: class index i in the .npy label files is CLASS_NAMES[i].
CLASS_NAMES = ["adipose", "background", "debris", "lymphocytes", "mucus",
               "smooth muscle", "normal colon mucosa", "cancer-associated stroma",
               "colorectal adenocarcinoma epithelium"]
NUM_CLASSES = len(CLASS_NAMES)
print("Keras:", keras.__version__, "| Backend:", keras.backend.backend())

## 1. Data loading, exploration and preprocessing

This project uses a fixed subset of PathMNIST, part of the MedMNIST v2 collection [1] and derived from the NCT-CRC-HE-100K colorectal histology patches [2]: 32,000 training and 8,000 test RGB tissue patches, 28×28 pixels each, with nine histopathological classes. Unlike a random image split, the supplied test partition is never used during model design, parameter search, early stopping, or preprocessing-statistic estimation.

In [ ]:
# Load the fixed train/test split (see README: Data). The split is loaded from files rather than
# re-downloaded so the results are reproducible.
# Images are uint8 arrays of shape (N, 28, 28, 3); labels are stored as (N, 1), so .ravel()
# flattens them to (N,) integer class indices.
X_full_raw = np.load("data/X_train.npy")
y_full = np.load("data/y_train.npy").ravel()
X_test_raw = np.load("data/X_test.npy")
y_test = np.load("data/y_test.npy").ravel()

print("Training images:", X_full_raw.shape, "| Test images:", X_test_raw.shape)
print("Pixel dtype:", X_full_raw.dtype, "| Range:", X_full_raw.min(), X_full_raw.max())
# Class balance: number of examples per class in the train and test sets, plus the train share (%).
class_counts = pd.DataFrame({
    "class": CLASS_NAMES,
    "train": np.bincount(y_full, minlength=NUM_CLASSES),
    "test": np.bincount(y_test, minlength=NUM_CLASSES),
})
class_counts["train_percent"] = (class_counts["train"] / len(y_full) * 100).round(1)
class_counts["test_percent"] = (class_counts["test"] / len(y_test) * 100).round(1)
display(class_counts)

### Data exploration

The supplied training split contains 2,728–4,697 images per class (8.5%–14.7% of the 32,000 images); the test split contains 682–1,174 per class. The most common training class is about 1.72 times the size of the least common. This is a moderate class-frequency difference, not a severe imbalance, so the experiments retain the original class distribution without rebalancing. The table and plot below compare both splits by class share.

The fixed-seed random examples are illustrative patches, not a systematic measurement of within-class variation. They show differences in colour and tissue texture that may be useful to a classifier; visual similarity between some examples is a reason to inspect the per-class results and confusion matrices, rather than evidence by itself that those classes are confused. We keep all three colour channels and spatial layout for the neural networks. The overall RGB summary in the next cell describes the dataset globally and should not be read as a class-specific colour analysis.


In [ ]:
# Show two reproducibly sampled examples per class; random sampling avoids relying on
# whichever examples happen to appear first in the files.
example_rng = np.random.default_rng(RANDOM_STATE)
fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(5.2, 13))
for label in range(NUM_CLASSES):
    class_indices = np.flatnonzero(y_full == label)
    example_indices = example_rng.choice(class_indices, size=2, replace=False)
    for column, index in enumerate(example_indices):
        axes[label, column].imshow(X_full_raw[index])
        axes[label, column].axis("off")
        if column == 0:
            axes[label, column].set_ylabel(CLASS_NAMES[label], rotation=0,
                                          ha="right", va="center", fontsize=8, labelpad=8)
plt.suptitle("PathMNIST: two fixed-seed random patches per class", fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_examples.png", dpi=150, bbox_inches="tight")
plt.show()

# Compare the class proportions in the supplied training and test splits.
fig, ax = plt.subplots(figsize=(8, 4.5))
class_counts.set_index("class")[["train_percent", "test_percent"]].plot(
    kind="barh", ax=ax, width=0.8)
ax.set_xlabel("Share of split (%)")
ax.set_ylabel("")
ax.set_title("Class proportions in the supplied splits")
ax.grid(axis="x", alpha=0.25)
ax.legend(title="Split")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Descriptive per-channel (R, G, B) pixel statistics on the raw 0-255 scale, computed in float64
# (float32 sums can drift). These are for exploration only; preprocessing statistics are estimated
# from the development split in the next cell.
print("RGB mean (0-255):", np.round(X_full_raw.mean(axis=(0, 1, 2), dtype=np.float64), 2))
print("RGB std (0-255):", np.round(X_full_raw.astype(np.float64).std(axis=(0, 1, 2)), 2))


### Preprocessing and validation design

First, split only the provided training data into **27,200 development examples (85%)** and **4,800 validation examples (15%)**, stratified by class. Then divide by 255 and estimate per-RGB-channel mean/standard deviation **from the 27,200 development examples only**, using float64 reductions to avoid float32 precision drift. Apply the same training-only transform to development, validation, and test images, with float32 stored inputs. Centring and scaling inputs to zero mean and unit variance is standard advice for gradient-based training [3].

Random Forest inputs use fixed 2×2 spatial average pooling and flattening (588 features), reducing tree-search cost; MLP/CNN keep 28×28×3 pixels. Standardisation is not necessary for trees but using these pooled standardised inputs is consistent and does not change threshold ordering. MLP/CNN augment during training with horizontal and vertical flips, a standard label-preserving augmentation [4]; histology patches have no privileged orientation. The test set is held out until Section 4.

**Runtime:** A 10,000-image stratified *development* subset is used for the Random Forest search and a further stratified 4,000-image subset is used for neural-network searches. Final training uses the full 27,200 development examples; the 4,800 validation images are reserved for early stopping and comparison. The subset constraint limits what differences between close trials mean.

In [ ]:
# Step 1: stratified 85/15 split of the provided training data into a development set (used to
# train) and a validation set (used to tune and early-stop). Splitting first means no statistic
# computed later can leak information from validation or test images. The test set is untouched.
X_dev_raw, X_val_raw, y_dev, y_val = train_test_split(
    X_full_raw, y_full, train_size=0.85,
    stratify=y_full, random_state=RANDOM_STATE)
# Step 2: stratified 10,000-image subset of the development data, used for the Random Forest search.
X_search10k_raw, _, y_search10k, _ = train_test_split(
    X_dev_raw, y_dev, train_size=10000,
    stratify=y_dev, random_state=RANDOM_STATE)

# Step 3: scale pixels from 0-255 to 0-1, then estimate the per-channel (R, G, B) mean and standard
# deviation from the development set only (in float64 to avoid precision drift).
X_dev_scaled = X_dev_raw.astype("float32") / 255.0
channel_mean = X_dev_scaled.mean(axis=(0, 1, 2), dtype=np.float64)
channel_std = X_dev_scaled.astype(np.float64).std(axis=(0, 1, 2))
assert np.all(channel_std > 0)

# The same transform is applied to every split: scale to 0-1, subtract the development-set channel
# mean and divide by its standard deviation (zero mean, unit variance per channel).
def standardise(images):
    return ((images.astype("float32") / 255.0 - channel_mean) / channel_std).astype("float32")

# Standardised images (float32, N x 28 x 28 x 3) used by the final MLP and CNN.
X_dev = standardise(X_dev_raw)
X_val = standardise(X_val_raw)
X_test = standardise(X_test_raw)
# Step 4: a smaller stratified 4,000-image subset (nested inside the 10,000 subset) keeps the
# neural-network searches fast.
X_search4k_raw, _, y_search4k, _ = train_test_split(
    X_search10k_raw, y_search10k, train_size=4000,
    stratify=y_search10k, random_state=RANDOM_STATE)
X_search4k = standardise(X_search4k_raw)

# Step 5: Random Forest features. 2x2 average pooling turns (N, 28, 28, 3) into (N, 14, 14, 3) by
# averaging each 2x2 block of pixels per channel; flattening gives 14 x 14 x 3 = 588 features.
def average_pool_images(images):
    # Split height and width into 14 blocks of 2 pixels each, then average within each block.
    n = len(images)
    return images.reshape(n, 14, 2, 14, 2, 3).mean(axis=(2, 4))

def flatten_images(images):
    return images.reshape(len(images), -1)

# Pooled RF features for the 10,000-image search subset (the 4,000-image subset is the neural-network one).
X_search10k_rf = flatten_images(average_pool_images(standardise(X_search10k_raw)))
X_val_rf = flatten_images(average_pool_images(X_val))
X_dev_rf = flatten_images(average_pool_images(X_dev))
X_test_rf = flatten_images(average_pool_images(X_test))

# Sanity checks: subset sizes here, and (at the end of the cell) a plausible red-channel mean.
assert len(X_search10k_rf) == len(y_search10k) == 10000
assert len(X_search4k) == len(y_search4k) == 4000

print("Dev / validation / 10k search / 4k search / test:",
      len(y_dev), len(y_val), len(y_search10k), len(y_search4k), len(y_test))
print("Channel means:", np.round(channel_mean, 4))
print("Channel std:", np.round(channel_std, 4))
print("RF features:", X_dev_rf.shape[1])
assert abs(float(channel_mean[0]) - 0.74) < 0.03, "Unexpected channel-mean value"

### Examples of preprocessed data
The figure below shows one development image per class at each stage: the raw patch, the standardised patch that the MLP and CNN receive, and the 2x2 average-pooled patch that the Random Forest receives (flattened to 588 features). Standardised values lie roughly in [-3, 3], so they are clipped and rescaled to [0, 1] for display only. This is why the colours look different from the raw images (each channel now has zero mean and unit variance). The printed statistics confirm the transform where the development set has mean 0 and standard deviation 1 per channel by construction, and the validation and test sets, transformed with the same development statistics, come out close to that. Random flips are applied inside the models during training only, so they are not shown here.

In [ ]:
# Show what the models actually receive for one development image per class: the raw patch, the
# standardised patch (MLP / CNN input) and the 2x2 average-pooled patch (Random Forest input).
def to_display(image):
    # Standardised values are roughly in [-3, 3]; clip and rescale to [0, 1] so matplotlib can draw them.
    return np.clip((image + 3.0) / 6.0, 0.0, 1.0)

preview_rng = np.random.default_rng(RANDOM_STATE)
column_titles = ["Raw\n28x28x3", "Standardised\n28x28x3\n(MLP / CNN input)", "2x2 pooled\n14x14x3\n(RF input)"]
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(5, 13))
for label in range(NUM_CLASSES):
    index = preview_rng.choice(np.flatnonzero(y_dev == label))
    images = [X_dev_raw[index],
              to_display(X_dev[index]),
              to_display(X_dev_rf[index].reshape(14, 14, 3))]
    for column, image in enumerate(images):
        ax = axes[label, column]
        ax.imshow(image)
        ax.set_xticks([])
        ax.set_yticks([])
        if label == 0:
            ax.set_title(column_titles[column], fontsize=8)
        if column == 0:
            ax.set_ylabel(textwrap.fill(CLASS_NAMES[label], 14), rotation=0,
                          ha="right", va="center", fontsize=8, labelpad=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "preprocessed_examples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Check the transform: per-channel mean and std in float64 (float32 reductions over this many pixels drift).
for split_name, split_images in [("development", X_dev), ("validation", X_val), ("test", X_test)]:
    print(f"{split_name:<11} shape {split_images.shape}  "
          f"mean {np.round(split_images.mean(axis=(0, 1, 2), dtype=np.float64), 3)}  "
          f"std {np.round(split_images.std(axis=(0, 1, 2), dtype=np.float64), 3)}")
print("Random Forest features (pooled and flattened):", X_dev_rf.shape)

## 2. Algorithm design and setup

This section defines one builder function per model and the helpers shared by the neural-network searches, then creates an example of each model before any tuning. The hyperparameters shown here are only defaults; the values used for the final models are chosen in Section 3.

### Random Forest

A Random Forest averages many decision trees, each trained on a bootstrap sample and restricted to a random subset of the features at every split [5]. Averaging decorrelated trees reduces the variance of a single deep tree, and the model needs little preprocessing, trains in parallel and gives a strong baseline on tabular data. It treats the input as an unordered feature vector, so it cannot use the spatial layout of an image directly. We therefore give it 2×2 average-pooled, flattened features (14×14×3 = 588), which also reduces the number of candidate splits at each node. `build_rf` fixes `random_state` and `n_jobs`; the hyperparameters searched in Section 3 are the number of trees, the maximum depth, `max_features` and the minimum leaf size.

In [ ]:
# Factory used by both the search and the final model, so forests are always built the same way
# (fixed seed for reproducibility, 4 parallel jobs).
def build_rf(n_estimators=100, max_depth=None, max_features="sqrt", min_samples_leaf=1, n_jobs=4):
    return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                  max_features=max_features,
                                  min_samples_leaf=min_samples_leaf, n_jobs=n_jobs,
                                  random_state=RANDOM_STATE)

# Example instance with default hyperparameters; the tuned values are chosen in Section 3.
example_rf = build_rf()
print("Example Random Forest:", example_rf)

### Fully connected neural network (MLP)

The MLP flattens each 28×28×3 image into 2,352 inputs, passes them through one or two hidden dense layers (ReLU or tanh activation, searched in Section 3) with dropout after each, and ends in a nine-way softmax. Dense layers learn nonlinear combinations of all pixels, but every weight is tied to one pixel position, so the model has many parameters and does not reuse a feature detector across the image. Dropout regularises these large layers. A random horizontal/vertical flip layer is active during training only, because tissue patches have no fixed orientation. Training uses backpropagation [6] with the Adam optimiser [8] and sparse categorical cross-entropy, since the labels are integer class indices.

In [ ]:
# MLP pipeline: image -> (random flips) -> flatten -> [Dense + Dropout] x depth -> 9-way softmax.
def build_mlp(hidden_units=(256, 128), dropout=0.3, learning_rate=0.001,
              activation="relu", use_augmentation=True):
    # Input: a 28x28 RGB image.
    layers = [keras.Input(shape=(28, 28, 3))]
    # Random horizontal/vertical flips; Keras applies them during training only.
    if use_augmentation:
        layers.append(keras.layers.RandomFlip("horizontal_and_vertical"))
    # Turn the image into a vector of 2,352 values, then add one Dense + Dropout pair per hidden layer.
    layers.append(keras.layers.Flatten())
    for units in hidden_units:
        layers.extend([keras.layers.Dense(int(units), activation=activation),
                       keras.layers.Dropout(float(dropout))])
    # Softmax output = probability of each of the 9 tissue classes.
    layers.append(keras.layers.Dense(NUM_CLASSES, activation="softmax"))
    model = keras.Sequential(layers)
    # Adam optimiser; sparse categorical cross-entropy because labels are integer class indices.
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=float(learning_rate)),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def hidden_units_for(width, depth):
    """MLP layer sizes for a sampled width and depth: (width,) or (width, width // 2)."""
    return (width,) if depth == 1 else (width, width // 2)

def build_mlp_from_search(parameters):
    """Build an MLP from one sampled configuration of the MLP search space."""
    return build_mlp(hidden_units=hidden_units_for(parameters["width"], parameters["depth"]),
                     dropout=parameters["dropout"], activation=parameters["activation"],
                     learning_rate=parameters["learning_rate"])

# Show the default architecture and its parameter count.
example_mlp = build_mlp()
example_mlp.summary()

### Convolutional neural network (CNN)

The CNN has two convolution + max-pooling blocks (the second with twice as many filters), followed by a flatten, dropout, a dense layer and a nine-way softmax. A convolution [7] shares one small kernel across the whole image, so a texture or nucleus detector works wherever it appears and needs far fewer parameters than a dense layer on raw pixels. Max-pooling halves the spatial size and adds some tolerance to small shifts. We use two blocks because the input is only 28×28: after two poolings the feature maps are 7×7, and a third would reduce them to 3×3. `padding="same"` keeps information at the image border and the activations are ReLU. Dropout before and after the dense layer regularises the classifier head. The CNN uses the same flip augmentation, loss and optimiser as the MLP, so differences between the two come from the architecture. Kernel size, filter count, dropout, dense width and learning rate are searched in Section 3.

In [ ]:
# CNN pipeline: image -> (random flips) -> [Conv + MaxPool] x 2 -> flatten -> Dropout -> Dense ->
# Dropout -> 9-way softmax.
def build_cnn(base_filters=32, kernel_size=3, dropout=0.3, dense_units=128,
              learning_rate=0.001, use_augmentation=True):
    # Input: a 28x28 RGB image.
    layers = [keras.Input(shape=(28, 28, 3))]
    # Random horizontal/vertical flips; Keras applies them during training only.
    if use_augmentation:
        layers.append(keras.layers.RandomFlip("horizontal_and_vertical"))
    layers.extend([
        # Block 1: base_filters local pattern detectors; pooling halves 28x28 to 14x14.
        keras.layers.Conv2D(int(base_filters), int(kernel_size), padding="same", activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        # Block 2: twice as many filters on the pooled maps; pooling halves 14x14 to 7x7.
        keras.layers.Conv2D(int(base_filters) * 2, int(kernel_size), padding="same", activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        # Classifier head: flatten the 7x7 maps, then Dropout -> Dense -> Dropout -> softmax.
        keras.layers.Flatten(),
        keras.layers.Dropout(float(dropout)),
        keras.layers.Dense(int(dense_units), activation="relu"),
        keras.layers.Dropout(float(dropout)),
        keras.layers.Dense(NUM_CLASSES, activation="softmax")])
    model = keras.Sequential(layers)
    # Same optimiser and loss as the MLP, so the architecture is the only difference.
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=float(learning_rate)),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Show the default architecture and its parameter count.
example_cnn = build_cnn()
example_cnn.summary()

### Helper for the neural-network hyperparameter searches

`run_neural_trial` trains one sampled configuration on a search subset and scores it on the validation set, so the MLP and CNN searches share the same early stopping, metrics and timing, and every trial uses the same random seed. The Random Forest search needs no helper beyond `build_rf`.

In [ ]:
def run_neural_trial(build_model, parameters, X_fit, y_fit,
                     epochs=10, batch_size=128, patience=2):
    """Train one sampled configuration and return its validation metrics and runtime.

    Every trial uses the same random seed, so trials differ only in their hyperparameters.
    """
    # Start from a clean Keras state and the same seed, then build the model for this configuration.
    keras.backend.clear_session()
    keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(parameters)
    # Train on the search subset. Early stopping watches validation accuracy: after `patience` epochs
    # without improvement it stops and restores the weights from the best epoch.
    start_time = time.perf_counter()
    history = model.fit(X_fit, y_fit, epochs=epochs, batch_size=batch_size,
                        validation_data=(X_val, y_val),
                        callbacks=[keras.callbacks.EarlyStopping(
                            monitor="val_accuracy", patience=patience,
                            restore_best_weights=True)], verbose=0)
    seconds = time.perf_counter() - start_time  # includes the per-epoch validation pass
    # Score the restored best model on the validation set (argmax of the probabilities = predicted class).
    predictions = np.argmax(model.predict(X_val, verbose=0), axis=1)
    # One results row: the hyperparameters plus metrics, runtime and the number of epochs actually run.
    return {**parameters, "val_accuracy": accuracy_score(y_val, predictions),
            "val_macro_f1": f1_score(y_val, predictions, average="macro"),
            "runtime_s": seconds, "epochs": len(history.epoch)}

## 3. Hyperparameter tuning

Each search evaluates validation accuracy and macro F1, records individual runtimes, and ranks by **validation accuracy** (then macro F1) only. No testing occurs here. We use a manageable grid for the Random Forest and an exhaustive grid for all three models (via `ParameterGrid`): 81 Random Forest, 72 MLP, and 108 CNN combinations. The Random Forest search trains on a stratified 10,000-image subset of the development data; the MLP and CNN searches train on a stratified 4,000-image subset nested within it (both subsets exist only to keep tuning fast). All searches are scored on the same 4,800-image validation set. Neural candidates use early stopping (patience 2) with up to ten epochs, restore the best validation weights, and share one random seed so that trials differ only in their hyperparameters.

For reproducibility, each search saves its results to `Results/*.csv`. With `RUN_SEARCHES = True` (see Setup) the searches are run; with `False` the saved CSVs are loaded instead, and a search also runs if its CSV is missing. The tuning visualisation cell uses the saved CSVs when all three are present; if search cells were skipped and any CSV is missing, it reports that the plots and summary were skipped. The final models use explicitly frozen hyperparameters and can still be trained without the search CSVs.

**Validation strategy: a single fixed hold-out.** Every candidate is trained on its search subset and scored on the same stratified validation set (15% of the provided training data; it never updates model weights and is used only to select hyperparameters and for early stopping). We chose this over k-fold cross-validation for four reasons: (i) a single held-out validation set is a standard, well-understood search strategy; (ii) k-fold would multiply the cost of every neural trial by k; (iii) 4,800 validation images give a standard error of roughly 0.4-0.7 percentage points on accuracy, which is enough to rank clearly different configurations; and (iv) one shared validation set keeps the tuning of the three models directly comparable. **Limitation:** each configuration is evaluated on one partition with one random seed, so differences of about one percentage point or less between configurations should not be read as meaningful. The test set plays no part in tuning.


### Random Forest search

Test the number of trees (variance and cost), maximum tree depth and minimum leaf size (both limit tree complexity and overfitting) and `max_features` (randomness/decorrelation). Because the grid is modest (81 combinations), evaluate every combination.

In [ ]:
# Grid search: every combination is tried (3 x 3 x 3 x 3 = 81 forests). max_depth=None lets trees
# grow until the leaves are pure; min_samples_leaf=3 forces larger, smoother leaves.
rf_grid = {"n_estimators": [200, 500, 1000], "max_depth": [10, 20, None],
           "max_features": ["sqrt", 0.3, 0.5], "min_samples_leaf": [1, 3, 5]}

# Run the search if requested (or if no saved results exist yet); otherwise load the saved results.
if RUN_SEARCHES or not RF_SEARCH_CSV.exists():
    rf_results = []
    for parameters in ParameterGrid(rf_grid):
        model = build_rf(**parameters)
        # Fit on the 10,000-image subset (timed), then score on the validation set.
        start_time = time.perf_counter()
        model.fit(X_search10k_rf, y_search10k)
        seconds = time.perf_counter() - start_time
        y_pred = model.predict(X_val_rf)
        rf_results.append({**parameters, "val_accuracy": accuracy_score(y_val, y_pred),
                           "val_macro_f1": f1_score(y_val, y_pred, average="macro"),
                           "runtime_s": seconds})
        print(parameters, "accuracy:", round(rf_results[-1]["val_accuracy"], 4))
    # Rank best-first by validation accuracy (ties broken by macro F1) and save to CSV.
    rf_results_df = pd.DataFrame(rf_results).sort_values(
        ["val_accuracy", "val_macro_f1"], ascending=False).reset_index(drop=True)
    rf_results_df.to_csv(RF_SEARCH_CSV, index=False)
else:
    rf_results_df = pd.read_csv(RF_SEARCH_CSV)

display(rf_results_df.round(4))

max_features = 0.5 is included to test whether considering more candidate features per split (stronger but more correlated trees) helps; min_samples_leaf ∈ {1, 3, 5} controls leaf size and hence how smooth, and how resistant to overfitting, each tree is.

### MLP search

Search over hidden-layer width/depth, dropout, activation and learning rate (five hyperparameters). Learning rate influences convergence, dropout controls regularisation, and width/depth change expressiveness and cost. Because the search uses a small 4,000-image subset, the full 72-combination grid is affordable (about two minutes in total), so every combination is evaluated.

In [ ]:
# Grid search over 5 hyperparameters: ParameterGrid tries every combination
# (3 widths x 2 depths x 2 dropouts x 2 activations x 3 learning rates = 72 models).
mlp_search_space = {
    "width": [128, 256, 512], "depth": [1, 2],
    "dropout": [0.2, 0.4], "activation": ["relu", "tanh"],
    "learning_rate": [0.0003, 0.001, 0.003]
}

# Run the search if requested (or if no saved results exist yet); otherwise load the saved results.
if RUN_SEARCHES or not MLP_SEARCH_CSV.exists():
    mlp_results = []
    mlp_trials = list(ParameterGrid(mlp_search_space))
    for trial_number, parameters in enumerate(mlp_trials, 1):
        mlp_results.append(run_neural_trial(build_mlp_from_search, parameters,
                                            X_search4k, y_search4k))
        print("MLP", trial_number, "/", len(mlp_trials), "accuracy:", round(mlp_results[-1]["val_accuracy"], 4))
    # Rank best-first by validation accuracy (ties broken by macro F1) and save to CSV.
    mlp_results_df = pd.DataFrame(mlp_results).sort_values(
        ["val_accuracy", "val_macro_f1"], ascending=False).reset_index(drop=True)
    mlp_results_df.to_csv(MLP_SEARCH_CSV, index=False)
else:
    mlp_results_df = pd.read_csv(MLP_SEARCH_CSV)

display(mlp_results_df.round(4))

### CNN search

Sample base filter count, kernel size, dropout, dense width and learning rate (five hyperparameters). More filters/wider kernels increase capacity and computational cost; dropout may reduce overfitting. Ten-epoch trials (with early stopping) on the 4,000-image search subset are intended to compare promising designs, not exhaustively optimise the architecture.

In [ ]:
# Grid search over 5 hyperparameters: ParameterGrid tries every combination
# (3 filter counts x 2 kernel sizes x 3 dropouts x 2 dense widths x 3 learning rates = 108 models).
cnn_search_space = {"base_filters": [16, 32, 48], "kernel_size": [3, 5],
                    "dropout": [0.2, 0.3, 0.4], "dense_units": [64, 128],
                    "learning_rate": [0.0003, 0.001, 0.003]}

# Run the search if requested (or if no saved results exist yet); otherwise load the saved results.
if RUN_SEARCHES or not CNN_SEARCH_CSV.exists():
    cnn_results = []
    cnn_trials = list(ParameterGrid(cnn_search_space))
    for trial_number, parameters in enumerate(cnn_trials, 1):
        cnn_results.append(run_neural_trial(lambda p: build_cnn(**p), parameters,
                                            X_search4k, y_search4k))
        print("CNN", trial_number, "/", len(cnn_trials), "accuracy:", round(cnn_results[-1]["val_accuracy"], 4))
    # Rank best-first by validation accuracy (ties broken by macro F1) and save to CSV.
    cnn_results_df = pd.DataFrame(cnn_results).sort_values(
        ["val_accuracy", "val_macro_f1"], ascending=False).reset_index(drop=True)
    cnn_results_df.to_csv(CNN_SEARCH_CSV, index=False)
else:
    cnn_results_df = pd.read_csv(CNN_SEARCH_CSV)

display(cnn_results_df.round(4))

### Tuning visualisation and summary

These three compact plots retain only what helps compare trial **accuracy against runtime**, while the full tables preserve the exact combinations.

In [ ]:
# The search cells save one CSV per model. They can be skipped when those saved results are
# available; if any file is missing, report that the tuning plots and summary were skipped.
search_csvs = {"Random Forest": RF_SEARCH_CSV, "MLP": MLP_SEARCH_CSV, "CNN": CNN_SEARCH_CSV}
missing_search_csvs = [str(path) for path in search_csvs.values() if not path.exists()]

if missing_search_csvs:
    print("Skipping tuning plots and summary; missing saved search results:",
          ", ".join(missing_search_csvs))
else:
    search_results = {name: pd.read_csv(path) for name, path in search_csvs.items()}

    # Colour each model by a hyperparameter that affects computational complexity.
    plot_groups = {
        "Random Forest": ("max_depth", [10, 20, None]),
        "MLP": ("depth", [1, 2]),
        "CNN": ("base_filters", [16, 32, 48])
    }
    colours = ["dodgerblue", "darkturquoise", "deeppink"]

    # One scatter plot per model: validation accuracy against training time shows the
    # accuracy/cost trade-off of each configuration.
    for name, results in search_results.items():
        fig, ax = plt.subplots(figsize=(6, 3.5))
        column, groups = plot_groups[name]
        values = pd.to_numeric(results[column], errors="coerce")

        for value, colour in zip(groups, colours):
            subset = results[values.isna() if value is None else values == value]
            ax.scatter(subset["runtime_s"], subset["val_accuracy"], s=48,
                       color=colour, label=str(value))

        ax.set_xlabel("Training time per configuration (s)")
        ax.set_ylabel("Validation accuracy")
        ax.set_title(name + " hyperparameter search")
        ax.legend(title=column, loc="lower right")
        ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / (name.lower().replace(" ", "_") + "_tuning.png"),
                    dpi=150, bbox_inches="tight")
        plt.show()

    # Summary table: number of trials, best validation accuracy and total search time per model.
    search_summary = pd.DataFrame({
        "model": list(search_results),
        "trials": [len(df) for df in search_results.values()],
        "best_val_accuracy": [df["val_accuracy"].max() for df in search_results.values()],
        "total_search_s": [df["runtime_s"].sum() for df in search_results.values()],
    })
    display(search_summary.round(3))
    search_summary.to_csv(RESULTS_DIR / "search_summary.csv", index=False)

## 4. Final models

Below, the **best validation-selected configurations are frozen as literal parameters**, copied from the top-ranked row of each search CSV. This makes these final training cells runnable even when the search cells are skipped. They are also checked against the saved results at runtime. Final models use all 27,200 development images and the untouched 4,800 validation images for early stopping, followed by one held-out evaluation on all 8,000 test images. Neural final training runs for up to 20 epochs with early stopping (patience 3) and best-weight restoration. Runtime is hardware-dependent.

In [ ]:
# Best hyperparameters, frozen as literals (copied from the top row of each search CSV) so the final
# models do not depend on the search cells having been run.
RF_BEST_PARAMS = {'n_estimators': 1000, 'max_depth': 20, 'max_features': 0.3, 'min_samples_leaf': 1}
MLP_BEST_PARAMS = {'hidden_units': (512, 256), 'dropout': 0.4, 'learning_rate': 0.0003, 'activation': 'relu', 'use_augmentation': True}
CNN_BEST_PARAMS = {'base_filters': 48, 'kernel_size': 3, 'dropout': 0.2, 'dense_units': 128, 'learning_rate': 0.001, 'use_augmentation': True}

# Check the frozen settings against the saved search results (skipped if the CSVs are absent).
if all(p.exists() for p in (RF_SEARCH_CSV, MLP_SEARCH_CSV, CNN_SEARCH_CSV)):
    rf_top = pd.read_csv(RF_SEARCH_CSV).iloc[0]   # rows are saved best-first
    mlp_top = pd.read_csv(MLP_SEARCH_CSV).iloc[0]
    cnn_top = pd.read_csv(CNN_SEARCH_CSV).iloc[0]
    # RF: compare each searched hyperparameter (max_depth is blank/NaN in the CSV when it was None).
    assert int(rf_top["n_estimators"]) == RF_BEST_PARAMS["n_estimators"]
    assert (None if pd.isna(rf_top["max_depth"]) else int(rf_top["max_depth"])) == RF_BEST_PARAMS["max_depth"]
    assert str(rf_top["max_features"]) == str(RF_BEST_PARAMS["max_features"])
    assert int(rf_top["min_samples_leaf"]) == RF_BEST_PARAMS["min_samples_leaf"]
    # MLP: rebuild the layer sizes from the saved width/depth, then compare the other settings.
    expected_hidden = hidden_units_for(int(mlp_top["width"]), int(mlp_top["depth"]))
    assert expected_hidden == MLP_BEST_PARAMS["hidden_units"]
    for key in ["dropout", "learning_rate"]:
        assert np.isclose(float(mlp_top[key]), MLP_BEST_PARAMS[key])
    assert mlp_top["activation"] == MLP_BEST_PARAMS["activation"]
    # CNN: all five searched hyperparameters must match.
    for key in ["base_filters", "kernel_size", "dropout", "dense_units", "learning_rate"]:
        assert np.isclose(float(cnn_top[key]), CNN_BEST_PARAMS[key])
    print("Validated: final hyperparameters match the top-ranked validation trials.")
else:
    print("Search CSVs not found: frozen hyperparameters were not cross-checked.")

In [ ]:
# Display the selected hyperparameters for each final model.
selected_params = {
    "Random Forest": RF_BEST_PARAMS,
    "MLP": MLP_BEST_PARAMS,
    "CNN": CNN_BEST_PARAMS
}

# One row per model, one column per hyperparameter; "-" marks a setting the model does not have.
# Building the table from the dicts (rather than filling NaNs afterwards) keeps whole numbers
# such as n_estimators as integers and avoids pandas fillna dtype warnings across versions.
param_names = list(dict.fromkeys(name for params in selected_params.values() for name in params))
selected_params_df = pd.DataFrame(
    [[params.get(name, "-") for name in param_names] for params in selected_params.values()],
    index=list(selected_params), columns=param_names, dtype=object)
display(selected_params_df)

### Final Random Forest

In [ ]:
# Final Random Forest: frozen best hyperparameters, trained on all 27,200 development images
# (pooled 588-feature version) and evaluated once on the held-out test set.
rf_final = build_rf(**RF_BEST_PARAMS)
# Training and prediction are timed separately.
start_time = time.perf_counter()
rf_final.fit(X_dev_rf, y_dev)
rf_train_time = time.perf_counter() - start_time
start_time = time.perf_counter()
rf_test_pred = rf_final.predict(X_test_rf)
rf_predict_time = time.perf_counter() - start_time
# Test metrics: accuracy and macro F1 (the unweighted mean of the per-class F1 scores).
rf_test_accuracy = accuracy_score(y_test, rf_test_pred)
rf_test_macro_f1 = f1_score(y_test, rf_test_pred, average="macro")
# Total number of tree nodes: a model-size measure for forests (parameter counts play this role for networks).
rf_total_nodes = sum(tree.tree_.node_count for tree in rf_final.estimators_)
print("Random Forest test accuracy:", round(rf_test_accuracy, 4),
      "| macro F1:", round(rf_test_macro_f1, 4),
      "| fit seconds:", round(rf_train_time, 2))

### Final MLP

In [ ]:
# Final MLP: frozen best hyperparameters, a fresh Keras state and the standard seed.
keras.backend.clear_session()
keras.utils.set_random_seed(RANDOM_STATE)
mlp_final = build_mlp(**MLP_BEST_PARAMS)
start_time = time.perf_counter()
# Train on the 27,200 development images (up to 20 epochs). The validation set is used only for
# early stopping: stop after 3 epochs without improvement and restore the best weights.
mlp_history = mlp_final.fit(
    X_dev, y_dev, epochs=20, batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                              restore_best_weights=True)], verbose=2)
mlp_train_time = time.perf_counter() - start_time
start_time = time.perf_counter()
# Predict the test set once (argmax of the class probabilities), then compute the test metrics.
mlp_test_pred = np.argmax(mlp_final.predict(X_test, verbose=0), axis=1)
mlp_predict_time = time.perf_counter() - start_time
mlp_test_accuracy = accuracy_score(y_test, mlp_test_pred)
mlp_test_macro_f1 = f1_score(y_test, mlp_test_pred, average="macro")
print("MLP test accuracy:", round(mlp_test_accuracy, 4),
      "| macro F1:", round(mlp_test_macro_f1, 4),
      "| fit seconds:", round(mlp_train_time, 2))

### Final CNN

In [ ]:
# Final CNN: frozen best hyperparameters, a fresh Keras state and the standard seed.
keras.backend.clear_session()
keras.utils.set_random_seed(RANDOM_STATE)
cnn_final = build_cnn(**CNN_BEST_PARAMS)
start_time = time.perf_counter()
# Train on the 27,200 development images (up to 20 epochs). The validation set is used only for
# early stopping: stop after 3 epochs without improvement and restore the best weights.
cnn_history = cnn_final.fit(
    X_dev, y_dev, epochs=20, batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                              restore_best_weights=True)], verbose=2)
cnn_train_time = time.perf_counter() - start_time
start_time = time.perf_counter()
# Predict the test set once (argmax of the class probabilities), then compute the test metrics.
cnn_test_pred = np.argmax(cnn_final.predict(X_test, verbose=0), axis=1)
cnn_predict_time = time.perf_counter() - start_time
cnn_test_accuracy = accuracy_score(y_test, cnn_test_pred)
cnn_test_macro_f1 = f1_score(y_test, cnn_test_pred, average="macro")
print("CNN test accuracy:", round(cnn_test_accuracy, 4),
      "| macro F1:", round(cnn_test_macro_f1, 4),
      "| fit seconds:", round(cnn_train_time, 2))

### Final comparison and class-wise mistakes

The final comparison includes predictive accuracy, macro F1, runtime and parameter/node counts. The confusion matrices are the only final-evaluation figure; per-class F1 is reported in a table to avoid a redundant chart.

In [ ]:
# Side-by-side table of the three final models: test accuracy and macro F1, fit/predict time, and
# size (Random Forest = total tree nodes; networks = parameter count).
model_comparison = pd.DataFrame([
    {"model": "Random Forest", "test_accuracy": rf_test_accuracy,
     "test_macro_f1": rf_test_macro_f1, "fit_s": rf_train_time,
     "predict_s": rf_predict_time, "parameters_or_nodes": rf_total_nodes},
    {"model": "MLP", "test_accuracy": mlp_test_accuracy,
     "test_macro_f1": mlp_test_macro_f1, "fit_s": mlp_train_time,
     "predict_s": mlp_predict_time, "parameters_or_nodes": mlp_final.count_params()},
    {"model": "CNN", "test_accuracy": cnn_test_accuracy,
     "test_macro_f1": cnn_test_macro_f1, "fit_s": cnn_train_time,
     "predict_s": cnn_predict_time, "parameters_or_nodes": cnn_final.count_params()},
])
model_comparison.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
display(model_comparison.round(4))

# Per-class F1 on the test set, treating each class against all the others (one-vs-rest).
per_class_f1 = pd.DataFrame({
    "class": CLASS_NAMES,
    "random_forest": f1_score(y_test, rf_test_pred,labels = range(NUM_CLASSES), average=None),
    "mlp": f1_score(y_test, mlp_test_pred, labels = range(NUM_CLASSES), average = None),
    "cnn": f1_score(y_test, cnn_test_pred, labels =  range(NUM_CLASSES), average = None),
})
per_class_f1.to_csv(RESULTS_DIR / "per_class_f1.csv", index=False)
display(per_class_f1.round(3))

# Row-normalised confusion matrices: each row shows how the images of one true class were spread
# over the predicted classes (rows sum to 1, so the diagonal is the per-class recall).
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import ConfusionMatrixDisplay

rose_wine = LinearSegmentedColormap.from_list(
    "RoseWine",
    [
        "#FFF8FB",  # Very light pink
        "#E7C6D6",  # Light rose
        "#B87E9E",  # MLP
        "#9B5277",  # CNN
        "#6A2246"   # Random Forest
    ]
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, pred) in zip(axes, [("Random Forest", rf_test_pred),
                                  ("MLP", mlp_test_pred), ("CNN", cnn_test_pred)]):
    confusion = confusion_matrix(y_test, pred, labels=range(NUM_CLASSES), normalize="true")
    ConfusionMatrixDisplay(confusion).plot(ax=ax, cmap=rose_wine, colorbar=False,
                                          values_format=".2f", xticks_rotation=90)
    ax.set_title(name)
    ax.set_xlabel("Predicted class (0-8)")
    ax.set_ylabel("Actual class (0-8)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

print("Class indices: " + "; ".join(f"{i}={name}" for i, name in enumerate(CLASS_NAMES)))

## 5. Interpretation and limitations

The interpretation of these results is given in the report. The CNN should exploit shared local texture filters more naturally than a flattened MLP; however, relative accuracy is a *measured* outcome, not a guarantee. Runtime figures reflect one CPU environment, and a single stratified validation partition plus a short sampled search cannot establish statistical significance. Training on the development split rather than refitting all 32,000 images is intentional: the held-out validation portion remains available for early stopping. These models are educational comparisons, not clinical diagnostic tools.

## 6. AI acknowledgement

Claude Sonnet 5.5 assisted with reviewing and revising the code and draft report.

## References

1. J. Yang *et al.*, “MedMNIST v2 — A large-scale lightweight benchmark for 2D and 3D biomedical image classification,” *Scientific Data*, vol. 10, art. 41, 2023.
2. J. N. Kather *et al.*, “Predicting survival from colorectal cancer histology slides using deep learning: A retrospective multicenter study,” *PLoS Medicine*, vol. 16, no. 1, e1002730, 2019.
3. Y. LeCun, L. Bottou, G. B. Orr and K.-R. Müller, “Efficient BackProp,” in *Neural Networks: Tricks of the Trade*, Springer, 1998, pp. 9–50.
4. C. Shorten and T. M. Khoshgoftaar, “A survey on image data augmentation for deep learning,” *Journal of Big Data*, vol. 6, art. 60, 2019.
5. L. Breiman, “Random forests,” *Machine Learning*, vol. 45, no. 1, pp. 5–32, 2001.
6. D. E. Rumelhart, G. E. Hinton and R. J. Williams, “Learning representations by back-propagating errors,” *Nature*, vol. 323, pp. 533–536, 1986.
7. Y. LeCun, L. Bottou, Y. Bengio and P. Haffner, “Gradient-based learning applied to document recognition,” *Proceedings of the IEEE*, vol. 86, no. 11, pp. 2278–2324, 1998.
8. D. P. Kingma and J. Ba, “Adam: A method for stochastic optimization,” *ICLR*, 2015.
9. F. Pedregosa *et al.*, “Scikit-learn: Machine learning in Python,” *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
10. F. Chollet *et al.*, Keras, 2015, https://keras.io.